<div style="background: linear-gradient(135deg, #1F3864 0%, #2E5FA3 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #a8c4e8; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #c8ddf5; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.12); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: #ddeeff; font-size: 1.05em; font-weight: 600;">&#128214; Chapter 7 &nbsp;&middot;&nbsp; Large Language Models</span>
  </div>
</div>
<div style="background: #f0f4fa; border-left: 5px solid #2E5FA3; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>What LLMs are, how they understand language, and how to use them through APIs to build real business applications &mdash; without training a single model yourself.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| 7.1 What Are LLMs? | From word2vec to Transformers · LLMs vs SLMs · Why this changed everything |
| 7.2 Tokenization and Embeddings | How text becomes numbers · BPE · Hugging Face tokenizers · Semantic similarity |
| 7.3 Using LLM APIs | First API call · Prompt engineering · Structured outputs |
| 7.4 Building a Chat App | Conversation history · System prompts · Gradio interface |
| 7.5 Ethical Considerations | Hallucination · Bias · Privacy · Cost |
| 7.6 Capstone: Chatting with Your Data | JSON earnings data · Trend analysis · Next-quarter prediction · Multi-turn conversation |

> **No local training in this chapter.** You will direct powerful pre-trained models through APIs.
> A free API key from OpenAI, Google Gemini, or Anthropic Claude is all you need.
> No GPU required.

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chapter 7 — Setup
# Installs everything this chapter needs.
# Expected time: 2–3 minutes on first run.
# ─────────────────────────────────────────────────────────────────────────────

!pip install --quiet transformers gradio openai google-generativeai anthropic

import os
import json
import textwrap
from transformers import AutoTokenizer

print("Setup complete. Ready to begin Chapter 7.")

---
## Where We Are Coming From

In Chapters 1 through 6, you built neural networks from scratch.

- In **Chapter 3**, you trained a regression network and saved it as a `.pth` file.
- In **Chapter 4**, you trained a classifier to predict customer churn.
- In **Chapter 5**, you trained a CNN to detect defects in product images.
- In **Chapter 6**, you trained an LSTM to forecast CO₂ levels weeks into the future.

Every time, the process was the same: collect data, design an architecture, run a training loop, save the weights.

**Chapter 7 is different.**

The models you will use here — Large Language Models — were trained by teams of hundreds of engineers on hundreds of billions of words, using thousands of GPUs, over months. You are not going to train them. You are going to *direct* them.

Think of the shift like this:

> **Chapters 1–6:** You baked the cake from scratch — flour, sugar, eggs, oven, timing.
>
> **Chapter 7:** A professional bakery already made the cake. Your job is to tell them exactly how you want it sliced, decorated, and served.

This is not a step down in skill. Knowing how to give clear, precise instructions to a powerful model — and knowing how to build applications around it — is one of the most valuable skills in business analytics today.

---
# 7.1 What Are Large Language Models?

## Starting simple: what does a language model do?

A **language model** is a system that has learned to predict what word comes next in a sentence.

That sounds almost too simple. But if you predict the next word well enough, across enough text, something remarkable happens: the model develops a deep working understanding of grammar, facts, logic, tone, and meaning — all without ever being told what any of those things are.

**Example:**

> "The capital of France is ___"

A language model that has seen enough text will fill in "Paris" with very high confidence. It has never been told the definition of "capital city." It learned the relationship from patterns in billions of sentences.

> "The meeting was cancelled because the CEO was ___ with the results."

Here the model might predict "unhappy", "dissatisfied", or "furious" — all contextually appropriate. It has learned how professional language works.

---

## A brief history — in plain language

```
Word2Vec (2013)
     |
     |  Words as vectors. "King" - "Man" + "Woman" ~ "Queen"
     |  Each word had ONE fixed representation.
     v
RNNs and LSTMs  (Chapter 6)
     |
     |  Text processed word-by-word with a hidden state.
     |  Good — but struggled with long documents.
     v
Transformers (2017 — "Attention Is All You Need")
     |
     |  Every word attends to every other word simultaneously.
     |  No recurrence — fully parallel. Scales beautifully.
     v
GPT, BERT, T5 (2018–2020)
     |
     |  Pre-trained on massive corpora.
     |  Fine-tuned on specific tasks.
     v
GPT-4 / Gemini / Claude / Llama (2020–present)
     |
     |  Hundreds of billions of parameters.
     |  Instruction-following, reasoning, coding,
     |  summarisation, translation — from one model.
     v
You are here.
```

---

## What makes an LLM "large"?

Three things working together:

| Ingredient | What it means | Example scale |
|-----------|---------------|---------------|
| **Parameters** | The learned numbers inside the model | GPT-4: estimated ~1 trillion |
| **Training data** | Text the model learned from | Books, websites, code, scientific papers |
| **Compute** | The GPU-hours needed to train it | Millions of dollars worth |

You do not need to understand attention mechanics to use an LLM effectively. But it is useful to know that the model has, in effect, read more text than any human ever could — and compressed what it learned into its weights.

---

## LLMs vs. SLMs — which one do you need?

| | **LLM** (Large Language Model) | **SLM** (Small Language Model) |
|-|-------------------------------|-------------------------------|
| **Examples** | GPT-4o, Gemini 1.5 Pro, Claude 3.5 | GPT-4o-mini, Gemini Flash, Phi-3 |
| **Strengths** | Complex reasoning, nuanced writing, coding | Speed, cost, privacy, on-device use |
| **Best for** | Multi-step analysis, creative tasks | Classification, extraction, simple Q&A |
| **Cost** | Higher per 1,000 tokens | Much lower |

**Rule of thumb:** Start with a smaller, cheaper model. If the quality is good enough, you are done. Only move to a larger model when the task genuinely requires it.

---

## The Five Main LLM Architectures

Not all LLMs are built the same way. Understanding these five families helps you
choose the right tool and make sense of news about new models.

---

### 1. Encoder-Only — The "Understanders"

**What they do:** Read text and build a deep understanding of it.
They do *not* generate new text — they analyse what is already there.

**How they work:** The model reads the entire input at once (left *and* right context
simultaneously) and produces a rich representation of each token's meaning.

```
Input:  "The bank by the river was steep."
         |      |    |    |      |    |
         v      v    v    v      v    v
      [vector][vector][vector]...[vector]   <- one rich embedding per token
```

**Business use cases:**
- Sentiment analysis: "Is this customer review positive or negative?"
- Document classification: "Which department should handle this email?"
- Semantic search: "Which of our 10,000 tickets is most similar to this new one?"

**Examples:** BERT, RoBERTa, DistilBERT

> **Key insight:** Encoder-only models are often smaller, faster, and cheaper than
> full LLMs. If your task is *understanding* — not generation — start here.

---

### 2. Decoder-Only — The "Generators"

**What they do:** Generate new text, one token at a time.
This is the architecture behind GPT-4, Claude, Gemini, and Llama.

**How they work:** The model reads the tokens so far and predicts the next one.
It can only attend to *previous* tokens — this is called **autoregressive** generation.

```
Input:    "The quarterly revenue"
               |
               v  predict next token
Output:   "The quarterly revenue declined"
               |
               v  predict next token
Output:   "The quarterly revenue declined unexpectedly"
               |
               v  ... and so on until done
```

**Business use cases:** Everything in Sections 7.3, 7.4, and 7.6 of this chapter —
drafting, summarising, classifying, Q&A, prediction.

**Examples:** GPT-4o, Claude 3.5, Gemini 1.5, Llama 3, Mistral

> **Key insight:** This is the architecture you interact with every time you use
> an LLM API. Decoder-only models are the default for business analytics work.

---

### 3. Encoder-Decoder — The "Translators"

**What they do:** Transform one piece of text into another.
Designed for tasks where input and output are different but related.

**How they work:** The encoder reads and compresses the input. The decoder generates
the output, attending to that compressed representation.

```
ENCODER                             DECODER
"Revenue sank last quarter"  -->  [compressed]  -->  "Les revenus ont baisse"
"The contract states that..." -->  [compressed]  -->  "Key clause: liability capped..."
```

**Business use cases:** Machine translation, document summarisation,
data-to-text generation (table of numbers to written commentary).

**Examples:** T5, BART, mT5

> **Key insight:** When your task is clearly "take this input, produce that output"
> with a fixed transformation, encoder-decoder models can be more efficient.

---

### 4. Mixture of Experts (MoE) — The "Specialists"

**What they do:** Handle a very large number of parameters without using all of them
on every input. A *router* selects which specialist sub-networks ("experts") to activate.

```
Input token  -->  Router  -->  selects 2 of 8 experts
                                    |
                         Expert 3 + Expert 7 process the token
                         (Experts 1,2,4,5,6,8 do nothing this turn)
                                    |
                              Output for this token
```

A model can have 400 billion total parameters but only *use* 50 billion for any
given input — giving the quality of a massive model at a fraction of the compute cost.

**Examples:** GPT-4 (rumoured), Mixtral 8x7B, Gemini 1.5 Pro

> **Key insight:** MoE is why some very large models are faster and cheaper than
> you might expect. You do not interact with them differently — the routing is invisible.

---

### 5. State Space Models (SSM) — The "Efficient Readers"

**What they do:** Process very long sequences efficiently. Transformers slow down
dramatically as text gets longer (cost grows quadratically). SSMs offer roughly linear cost.

```
Transformer (attention):  cost grows as O(n squared) with sequence length
  1,000 tokens:  1M operations
  10,000 tokens: 100M operations  <- expensive

SSM:                      cost grows as O(n) with sequence length
  1,000 tokens:  1K operations
  10,000 tokens: 10K operations   <- efficient
```

In spirit, an SSM is similar to the LSTM hidden state from Chapter 6 — it compresses
history into a compact fixed-size state — but with more sophisticated mathematics.

**Business use cases:** Analysing very long documents (100-page contracts, full books,
long transcripts) where inputs regularly exceed typical Transformer context windows.

**Examples:** Mamba, S4, Jamba (hybrid Transformer + Mamba)

> **Key insight:** SSMs are an active research frontier. For most business tasks today,
> decoder-only Transformers remain the default. SSMs become compelling when inputs are
> consistently very long.

---

### At a glance

| Architecture | Reads | Generates | Best for | Examples |
|-------------|-------|-----------|----------|----------|
| **Encoder-Only** | Full input, both directions | No | Understanding, classification, search | BERT, RoBERTa |
| **Decoder-Only** | Left context only | Yes | Generation, Q&A, chat, coding | GPT-4o, Claude, Gemini |
| **Encoder-Decoder** | Full input (encoder) | Yes | Translation, summarisation | T5, BART |
| **MoE** | Varies (built on Dec/Enc-Dec) | Varies | Large-scale efficiency | Mixtral, Gemini 1.5 |
| **SSM** | Compressed state | Yes | Very long sequences | Mamba, Jamba |

---
### 💡 Illustration: Why Attention Changed Everything

Imagine you are reading the sentence:

> *"The bank by the river was steep, so the fisherman tied his boat to the **bank**."*

The second "bank" means riverbank, not financial institution. How do you know?
Because you attended to "river", "fisherman", and "boat" earlier in the sentence.

This is exactly what a Transformer's **attention mechanism** does — for every word,
it learns to look at every other word and decide which ones are most relevant.

```
Sentence: "The bank by the river was steep"

When processing "bank", the model attends to:

    "The"    --> low attention  (0.02)
    "bank"   --> self           (0.15)
    "by"     --> low            (0.03)
    "the"    --> low            (0.02)
    "river"  --> HIGH attention (0.61)  <-- this is the key signal
    "was"    --> low            (0.05)
    "steep"  --> medium         (0.12)
```

By attending strongly to "river", the model learns that this "bank" is a geographical
feature, not a financial one.

An LSTM (Chapter 6) had to carry this context forward one step at a time.
A Transformer reads the whole sentence at once and figures out which parts matter.
That is why Transformers work better on long documents.

---

### How the attention scores are calculated — the role of Softmax

You may notice the attention scores above sum to exactly 1.0.
That is not a coincidence. The raw scores are passed through the **softmax function**
before being used — the same softmax you encountered in the classification chapters.

**What softmax does to raw attention scores:**

```
Raw scores (before softmax):
  "The"   -->  -2.1   (low relevance)
  "bank"  -->   0.8   (moderate)
  "river" -->   3.4   (highly relevant)
  "steep" -->   0.6   (moderate)

After softmax (converts to probabilities that sum to 1.0):
  "The"   -->  0.02   ( 2%)
  "bank"  -->  0.15   (15%)
  "river" -->  0.61   (61%)   <-- most attention
  "steep" -->  0.12   (12%)
  others  -->  0.10   (10%)   <-- remaining words share the rest
```

**Why softmax? Two reasons:**

1. **Produces a probability distribution.** Every score is between 0 and 1,
   and they all sum to 1. The scores become interpretable as "how much attention
   to pay to each word."

2. **Amplifies differences.** A raw score of 3.4 vs 0.8 becomes 61% vs 15%.
   The highest-scoring word gets a disproportionately large share of the attention.
   The model learns to *focus*, not just *prefer slightly*.

> **The connection to what you already know:** In Chapter 4 (FFN classifier),
> softmax in the output layer converted raw scores into class probabilities.
> In attention, softmax does the same thing — but over *positions in a sentence*
> rather than over class labels. Same function, new role.

### 📝 Exercise 7.1 — LLMs in Your Organisation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 7.1
#
# Think about the organisation or industry you work in (or want to work in).
# Identify THREE tasks where an LLM could save 10+ hours per week.
#
# For each task, write:
#   Task       : what the person currently does manually
#   LLM action : what the LLM would do instead
#   Time saved : rough estimate per week
#   Risk       : one thing that could go wrong
#
# Example:
#   Task       : A lawyer reads 50 contracts per week looking for unusual clauses
#   LLM action : LLM flags unusual clauses with a brief explanation
#   Time saved : ~15 hours/week
#   Risk       : LLM misses a subtle clause; lawyer still needs to review
#
# Task 1:
#   Task       : ?
#   LLM action : ?
#   Time saved : ?
#   Risk       : ?
#
# Task 2:
#   Task       : ?
#   LLM action : ?
#   Time saved : ?
#   Risk       : ?
#
# Task 3:
#   Task       : ?
#   LLM action : ?
#   Time saved : ?
#   Risk       : ?
# ─────────────────────────────────────────────────────────────────────────────
print("Exercise 7.1 complete.")

---
# 7.2 Tokenization and Embeddings

## How does text become numbers?

Neural networks work with numbers. Text is made of letters and words.
The conversion happens in two steps: **tokenization** then **embeddings**.

---

## Step 1: Tokenization — cutting text into pieces

A **token** is the basic unit that an LLM works with. Tokens are not always full words:

- A complete word: `"business"` → one token
- Part of a word: `"tokenization"` → might split into `["token", "ization"]`
- A punctuation mark: `"."` → one token

**Why split words?** A vocabulary of 50,000 tokens can cover almost any English text
by combining common sub-word pieces — including rare words, names, and technical terms.

---

### 💡 Illustration: Tokenization in action

```
Input: "The quarterly revenue declined unexpectedly."

Tokens:  ["The", "quarterly", "revenue", "declined", "unexpected", "##ly", "."]
IDs:     [  464,     18742,     7168,      3830,        8320,       ##ly,   13]

A less common word splits further:
  "unexpectedly" --> ["un", "expect", "ed", "ly"]   (4 tokens, not 1)
```

**Important for business users:** You are often charged per token.
Rough rule: 1 token ≈ 0.75 English words. 1,000 tokens ≈ one page of text.

---

## Step 2: Embeddings — turning tokens into meaning

After tokenization, each token gets converted to a **vector** — a list of numbers.

You saw this idea in Chapter 6 when we gave each of 52 weeks its own 8-number
vector using `nn.Embedding`. LLMs do the same for words, at much larger scale:

```
Chapter 6 (week embeddings):
   week_index --> 8-number vector    (52 possible weeks)

LLM (word embeddings):
   token_id   --> 768-number vector  (50,000 possible tokens)
               or 4096-number vector for larger models
```

The key property: **similar meanings end up close together in space.**

```
"revenue"  is close to  "income", "sales", "earnings"
"revenue"  is far from  "weather", "bicycle", "ocean"

"declined" is close to  "dropped", "fell", "decreased"
"declined" is far from  "improved", "grew", "increased"
```

This means the model understands that "sales fell" and "revenue declined"
are saying roughly the same thing — even though no word is shared.

### Tokenizing business text with Hugging Face

Let us see tokenization in action using a real tokenizer.
We use `bert-base-uncased` — a well-known model. Only the vocabulary file
is downloaded (~500 KB), not the full model.

One thing to watch for: when a word is entirely absent from the vocabulary,
the tokenizer replaces it with **`[UNK]`** (short for *unknown*).
This is rare with modern sub-word tokenizers — most words can be broken into
known pieces — but it does appear for very unusual character sequences, symbols,
or highly specialised technical codes. The code below includes an example.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tokenization: watching text become token IDs
# Only the vocabulary is downloaded -- not the full model.
# ─────────────────────────────────────────────────────────────────────────────

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sentences = [
    "The quarterly revenue declined unexpectedly.",
    "Our customer churn rate increased by 12 percent.",
    "The CEO announced a strategic restructuring plan.",
    "Tokenization splits uncommon words like synergistic carefully.",
    # This string contains a made-up code that BERT has never seen.
    # Watch for [UNK] in the token output -- BERT's way of saying
    # "I have no sub-word pieces for this character sequence."
    "The compound XZYQ-4471a showed unusual pharmacokinetic properties.",
]

print("=" * 65)
print("Tokenization Examples")
print("=" * 65)

BERT_UNK_ID = 100   # BERT's vocabulary ID for [UNK]

for sentence in sentences:
    tokens    = tokenizer.tokenize(sentence)
    token_ids = tokenizer.encode(sentence, add_special_tokens=False)
    has_unk   = "[UNK]" in tokens or BERT_UNK_ID in token_ids

    print(f"\nOriginal : {sentence}")
    print(f"Tokens   : {tokens}")
    print(f"IDs      : {token_ids}")
    print(f"Count    : {len(tokens)} tokens  |  Words: {len(sentence.split())}", end="")
    if has_unk:
        print("  <<< contains [UNK] -- word not in vocabulary", end="")
    print()
    print("-" * 65)

print()
print("Key observations:")
print("  - Common words are usually 1 token each.")
print("  - Long or uncommon words split into sub-word pieces (shown with ## prefix).")
print("  - Punctuation becomes its own token.")
print("  - [UNK] appears when a character sequence is entirely outside the vocabulary.")
print("  - Modern GPT-style tokenizers (BPE) rarely produce [UNK] -- they fall back")
print("    to byte-level pieces so any UTF-8 character can be encoded.")

### 📝 Exercise 7.2 — Explore Your Domain's Vocabulary

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 7.2 — Tokenize domain-specific business text
#
# Replace the sentences below with text from YOUR industry or field.
# Examples to try:
#   Legal   : "The indemnification clause supersedes the arbitration agreement."
#   Medical : "The patient exhibited tachycardia and diaphoresis post-operatively."
#   Finance : "The collateralised debt obligation was marked to market at par."
#   Tech    : "The microservices architecture enables horizontal scalability."
#
# After running, reflect:
#   1. Which words from your domain split into multiple tokens?
#   2. Why might this matter when estimating API costs?
#   3. Are there terms the tokenizer handles well vs. poorly?
# ─────────────────────────────────────────────────────────────────────────────

my_sentences = [
    "Replace this with a sentence from your industry.",
    "Add a second sentence with domain-specific terminology.",
    "Try a sentence with abbreviations like KPI, ROI, or EBITDA.",
]

print("=" * 60)
print("Your Domain Tokenization")
print("=" * 60)

for sentence in my_sentences:
    tokens = tokenizer.tokenize(sentence)
    print(f"\nSentence : {sentence}")
    print(f"Tokens   : {tokens}")
    print(f"Count    : {len(tokens)} tokens from {len(sentence.split())} words")
    print("-" * 60)

---
# 7.3 Using LLM APIs

## What is an API?

An **API** (Application Programming Interface) is a way for your code to talk to
another service over the internet.

When you make an LLM API call, you are saying:

> "Here is my text (the prompt). Please process it and send back the response."

You send text. You receive text back. No training, no GPU, no model files.

---

## The anatomy of an LLM API call

```
API CALL STRUCTURE
==================

1. MODEL       Which model to use
               "gpt-4o-mini", "gemini-1.5-flash", etc.

2. MESSAGES    The conversation so far
               role: "system"    --> instructions about how to behave
               role: "user"      --> what you are asking
               role: "assistant" --> what the model said previously

3. PARAMETERS
               temperature = 0   --> focused, deterministic
               temperature = 1   --> more creative and varied
               max_tokens        --> maximum length of the response
```

---

## Prompt engineering: the skill of giving good instructions

The quality of an LLM's output depends almost entirely on how clearly you communicate.
A well-structured prompt has up to seven components. You do not need all seven every time,
but knowing each one lets you diagnose *why* an output is poor and fix it precisely.

---

### The 7-Element Prompt Framework

```
+------------------+------------------------------------------------------+
| Element          | What it does                                         |
+------------------+------------------------------------------------------+
| 1. ROLE          | Who the model should be                              |
| 2. CONTEXT       | Background the model needs to answer well            |
| 3. TASK / ACTION | Exactly what you want the model to do                |
| 4. INPUT DATA    | The actual content to work on                        |
| 5. CONSTRAINTS   | Rules, limits, and things to avoid                   |
| 6. FORMAT        | How the output should be structured                  |
| 7. EXAMPLES      | Demonstrations of what good output looks like        |
+------------------+------------------------------------------------------+
```

A simple factual question needs very few of these.
A complex classification or generation task benefits from all of them.

---

### Element 1: Role

Giving the model a role activates relevant patterns from its training.

```
WITHOUT ROLE:
"Is this contract clause problematic?"

WITH ROLE:
"You are a senior contracts analyst at a technology company specialising
in SaaS vendor agreements. Review the clause below and flag any terms
that could increase the company's liability."
```

The role does not need to match a real job title.
"You are a plain-English explainer for non-technical executives" is valid.

---

### Element 2: Context

Context provides facts the model cannot know on its own —
your company, your audience, the situation, relevant history.

```
WITHOUT CONTEXT:
"Write a response to this customer complaint."

WITH CONTEXT:
"You are responding on behalf of NovaPay, a B2B payments startup.
Our refund policy allows full refunds within 14 days. The customer
below has been a subscriber for 2 years. Tone: warm and professional."
```

Context prevents the model from making assumptions that do not fit your situation.

---

### Element 3: Task / Action

State the task as a clear, specific verb phrase.
Vague verbs produce vague outputs.

```
VAGUE VERBS:    "Help me with...", "Look at...", "Think about..."

PRECISE VERBS:  "Summarise", "Classify", "Extract", "Compare",
                "Rewrite", "Translate", "List", "Score", "Flag"

VAGUE:    "Help me with this email chain."

PRECISE:  "Extract all action items from this email chain.
           For each item, identify: the person responsible,
           the deadline (if stated), and the priority (high/medium/low)."
```

---

### Element 4: Input Data

When your prompt contains data, always separate it clearly from your instructions
using delimiters so the model never confuses instructions with content.

```
WITHOUT DELIMITER (confusing):
"Summarise the following: Our Q3 revenue was $248M..."

WITH DELIMITER (clear):
"Summarise the following earnings excerpt in 3 bullet points.

--- EXCERPT START ---
Our Q3 revenue was $248M, representing 18% YoY growth...
--- EXCERPT END ---"
```

Common delimiters: `---`, `===`, triple backticks, or XML-style tags.

---

### Element 5: Constraints

Constraints tell the model what *not* to do as well as what *to* do.

```
"Summarise this earnings transcript.

Constraints:
- Maximum 5 bullet points
- Include exact figures wherever they appear in the transcript
- Do not speculate about causes -- only report what was stated
- Do not mention any competitor by name
- If a metric is not mentioned, omit it rather than estimating"
```

Constraints are especially important for length, tone, accuracy, and safety.

---

### Element 6: Format

Specify the output format explicitly.
Options: plain text, bullet list, numbered list, JSON, Markdown table, CSV.

```
WITHOUT FORMAT:
"Analyse these three products."

WITH FORMAT:
"Analyse these three products and return a Markdown table with columns:
 Product Name | Key Strength | Key Weakness | Recommended Audience

 One row per product. Keep each cell to one sentence."
```

For programmatic use (when you will parse the output in code), request JSON
and specify the exact schema:

```
"Return ONLY a JSON object with these exact keys:
 {
   'sentiment':   'positive' | 'neutral' | 'negative',
   'confidence':  0.0 to 1.0,
   'key_phrase':  'the phrase that most signals the sentiment'
 }
 No explanation. No markdown. Only the JSON object."
```

---

### Element 7: Examples (Few-shot prompting)

When a task is ambiguous or requires a specific style, showing examples
is more reliable than describing the task. This is called **few-shot prompting**.

```
"Classify the sentiment of customer reviews.

Examples:
  Review:    'The onboarding was seamless and support responded in minutes.'
  Sentiment: positive

  Review:    'The product does what it says, nothing more.'
  Sentiment: neutral

  Review:    'Three outages in one month. We are evaluating alternatives.'
  Sentiment: negative

Now classify:
  Review: 'The new dashboard is faster, though export still crashes occasionally.'"
```

Examples calibrate the model's judgment — especially useful when category
definitions have domain-specific nuance.

---

### A complete prompt using all 7 elements

```
[ROLE]
You are a senior legal analyst specialising in SaaS vendor contracts.

[CONTEXT]
Our company is a mid-size healthcare technology firm evaluating a new
cloud storage agreement. We must comply with HIPAA and EU data residency rules.

[TASK]
Review the contract clause below and provide a risk assessment.

[INPUT DATA]
--- CLAUSE START ---
VendorX reserves the right to transfer Customer data to third-party
sub-processors outside the EEA, provided VendorX implements appropriate
safeguards as determined by VendorX.
--- CLAUSE END ---

[CONSTRAINTS]
- Focus only on data residency and privacy risks
- Do not speculate about VendorX's intentions
- Flag any terms that give VendorX unilateral discretion

[FORMAT]
Return:
1. Risk level: Low / Medium / High
2. Key concern (1 sentence)
3. Specific phrase that creates the risk (quote from clause)
4. Recommended action (1 sentence)

[EXAMPLE]
Risk level: High
Key concern: The vendor has sole discretion to determine compliance adequacy.
Specific phrase: "as determined by VendorX"
Recommended action: Replace with a named EU-approved standard (e.g. EU SCCs).
```

---

### Quick reference: prompt quality checklist

```
[ ] Have I given the model a clear role?
[ ] Does it have the background context it needs?
[ ] Is my task a precise action verb?
[ ] Is my input data clearly separated from my instructions?
[ ] Have I stated constraints (length, tone, accuracy, safety)?
[ ] Have I specified the output format?
[ ] For complex tasks: have I included at least one example?
[ ] Temperature: 0.0-0.3 for factual tasks, 0.7+ for creative tasks?
```

---

## Setting up your API key

Choose any one of the following — all have free tiers sufficient for this chapter:

| Provider | Get a free key at | Model to use |
|----------|------------------|--------------|
| OpenAI | platform.openai.com | `gpt-4o-mini` |
| Google Gemini | aistudio.google.com | `gemini-1.5-flash` |
| Anthropic Claude | console.anthropic.com | `claude-haiku-4-5` |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# API Key Setup -- using Colab Secrets (recommended)
#
# Your API key is a password. Never paste it directly into a notebook cell
# that you might share, commit to GitHub, or submit as coursework.
#
# SECURE METHOD -- Colab Secrets (same pattern as the Kaggle token in Chapter 2):
#   1. Click the key icon in the left sidebar of Google Colab
#   2. Click "+ Add new secret"
#   3. Name:   OPENAI_API_KEY  (or GEMINI_API_KEY / ANTHROPIC_API_KEY)
#   4. Value:  paste your actual key
#   5. Toggle "Notebook access" to ON
#   6. Run this cell -- it reads the key from Secrets, never from your code
#
# The key is stored in your Google account, not in the notebook file.
# You can share the notebook safely without exposing the key.
# ─────────────────────────────────────────────────────────────────────────────

import os

# ── STEP 1: Choose your provider ─────────────────────────────────────────────
PROVIDER = "openai"   # options: "openai"  |  "gemini"  |  "anthropic"

# ── STEP 2: Load the key from Colab Secrets ──────────────────────────────────
try:
    from google.colab import userdata

    secret_map = {
        "openai":    ("OPENAI_API_KEY",    "gpt-4o-mini"),
        "gemini":    ("GEMINI_API_KEY",    "gemini-1.5-flash"),
        "anthropic": ("ANTHROPIC_API_KEY", "claude-haiku-4-5"),
    }
    secret_name, MODEL_NAME = secret_map[PROVIDER]
    API_KEY = userdata.get(secret_name)
    print(f"Key loaded from Colab Secrets  ({secret_name})")

except Exception:
    # ── Fallback: running locally (not on Colab) ──────────────────────────────
    # Set your key as an environment variable BEFORE launching Jupyter:
    #
    #   Mac/Linux:  export OPENAI_API_KEY="sk-..."
    #   Windows:    $env:OPENAI_API_KEY="sk-..."
    #
    # Never hardcode your key in the cell below.
    env_map = {
        "openai":    ("OPENAI_API_KEY",    "gpt-4o-mini"),
        "gemini":    ("GEMINI_API_KEY",    "gemini-1.5-flash"),
        "anthropic": ("ANTHROPIC_API_KEY", "claude-haiku-4-5"),
    }
    env_var, MODEL_NAME = env_map[PROVIDER]
    API_KEY = os.environ.get(env_var, "")

    if API_KEY:
        print(f"Key loaded from environment variable  ({env_var})")
    else:
        print(f"WARNING: No key found for provider '{PROVIDER}'.")
        print(f"  On Colab : add '{env_var}' to Secrets (key icon in sidebar).")
        print(f"  Locally  : run  export {env_var}=your-key  before Jupyter.")
        print("  API calls will fail until a key is provided.")

# ── STEP 3: Initialise the client ─────────────────────────────────────────────
from openai import OpenAI

if PROVIDER == "openai":
    client = OpenAI(api_key=API_KEY)

elif PROVIDER == "gemini":
    # Gemini supports the OpenAI-compatible endpoint
    client = OpenAI(
        api_key=API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )

elif PROVIDER == "anthropic":
    # Anthropic also supports an OpenAI-compatible endpoint
    client = OpenAI(
        api_key=API_KEY,
        base_url="https://api.anthropic.com/v1/"
    )

print(f"Client ready.  Model: {MODEL_NAME}")
print("All API calls in this chapter use the 'client' and 'MODEL_NAME' variables.")

### Your first API call

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# First API call -- a simple business question
# Read through every comment: the pattern is explained line by line.
# ─────────────────────────────────────────────────────────────────────────────

from openai import OpenAI

client = OpenAI(api_key=API_KEY)

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        # System message: tells the model HOW to behave
        {
            "role": "system",
            "content": (
                "You are a helpful business analyst. "
                "Give concise, practical answers. "
                "Use bullet points when listing multiple items."
            )
        },
        # User message: the actual question
        {
            "role": "user",
            "content": "What are three early warning signs that a company's customer churn rate is about to increase?"
        }
    ],
    temperature=0.3,   # low = more focused and consistent
    max_tokens=300,
)

answer = response.choices[0].message.content

print("=" * 60)
print("Model response:")
print("=" * 60)
print(answer)
print()
print(f"Tokens -- Prompt: {response.usage.prompt_tokens} | "
      f"Response: {response.usage.completion_tokens} | "
      f"Total: {response.usage.total_tokens}")

### Structured outputs — getting JSON back

In business applications, you often want the model to return structured data
(a category, a score, a list of fields) rather than free-form prose.
You can do this by being explicit in your prompt.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Structured output: classifying customer complaints as JSON
# temperature=0.0 for maximum consistency in classification tasks
# ─────────────────────────────────────────────────────────────────────────────

import json

customer_complaints = [
    "My order arrived three weeks late and the packaging was destroyed.",
    "I have been charged twice for the same subscription this month.",
    "Your app keeps crashing every time I try to view my account balance.",
    "The product works fine but I could not find the return policy anywhere.",
]

CLASSIFICATION_SYSTEM_PROMPT = (
    "You are a customer service analyst. "
    "Classify each complaint and respond ONLY with a JSON object. "
    "No explanation, no extra text -- only the JSON. "
    "Format: "
    '{"category": "delivery|billing|technical|information", '
    '"urgency": "low|medium|high", '
    '"sentiment": "neutral|frustrated|angry", '
    '"summary": "one sentence"}'
)

print("=" * 60)
print("Classifying customer complaints")
print("=" * 60)

for complaint in customer_complaints:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": CLASSIFICATION_SYSTEM_PROMPT},
            {"role": "user",   "content": complaint}
        ],
        temperature=0.0,
        max_tokens=150,
    )

    raw = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(raw)
        print(f"\nComplaint : {complaint[:65]}...")
        print(f"Category  : {parsed.get('category', 'n/a')}")
        print(f"Urgency   : {parsed.get('urgency', 'n/a')}")
        print(f"Sentiment : {parsed.get('sentiment', 'n/a')}")
        print(f"Summary   : {parsed.get('summary', 'n/a')}")
    except json.JSONDecodeError:
        print(f"\nComplaint : {complaint[:65]}")
        print(f"Raw output: {raw}")
        print("(Could not parse as JSON -- check prompt.)")

    print("-" * 60)

### 📝 Exercise 7.3 — Compare Two Temperature Settings

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 7.3 -- Compare temperature=0.0 vs temperature=0.9
#
# Use the same prompt twice with different temperature settings.
# After running, reflect:
#   1. Did both runs make the same key points?
#   2. Which felt more reliable for a business audience?
#   3. For a safety-critical task, which temperature would you choose?
# ─────────────────────────────────────────────────────────────────────────────

business_prompt = (
    "A mid-size retail company is considering replacing its human customer "
    "service team (20 agents) with an LLM-powered chatbot. "
    "What are the three most important risks they should evaluate? "
    "Respond in 3 numbered points, each 2-3 sentences."
)

print("Run A -- temperature 0.0 (focused):")
print("-" * 50)
r_a = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": business_prompt}],
    temperature=0.0, max_tokens=400,
)
print(r_a.choices[0].message.content)

print("\n\nRun B -- temperature 0.9 (more varied):")
print("-" * 50)
r_b = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": business_prompt}],
    temperature=0.9, max_tokens=400,
)
print(r_b.choices[0].message.content)

---
# 7.4 Building a Chat App

## Why conversation history matters

An LLM has **no memory between separate API calls**. Every call starts fresh.

You give it a conversation history by including all previous messages in the
`messages` list of each new call:

```
Turn 1 -- You send:
  messages = [
    {"role": "user", "content": "What is churn rate?"}
  ]

Turn 2 -- You send:
  messages = [
    {"role": "user",      "content": "What is churn rate?"},     <- previous
    {"role": "assistant", "content": "Churn rate is the..."},    <- previous
    {"role": "user",      "content": "How do I reduce it?"}      <- new
  ]
```

Each API call sends the full conversation. The model reads it all and
responds in context. This is how every chatbot is built.

---

## The system prompt: your model's personality and rules

The **system prompt** (`role: "system"`) appears at the start of every
conversation. It tells the model its job, tone, and limits.

Example system prompts:

```
# Financial report summariser:
"You are a financial analyst assistant. Summarise documents clearly.
Always include key numbers. Flag risks. Do not make recommendations."

# Customer service agent:
"You are a friendly representative for TechCorp. You can answer questions
about our products. If you do not know, say so and offer to escalate."

# Internal HR assistant:
"Answer questions about company policy based only on the provided documents.
If a question falls outside policy, say: I do not have policy on that topic."
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# A multi-turn chat loop
#
# We keep a running history list. Each new message is appended, then
# the full history is sent with every API call. This is the production pattern.
# ─────────────────────────────────────────────────────────────────────────────

def chat(system_prompt, user_questions):
    history = [{"role": "system", "content": system_prompt}]

    print("=" * 60)
    print("Chat Session")
    print("=" * 60)

    for question in user_questions:
        history.append({"role": "user", "content": question})

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=history,
            temperature=0.4,
            max_tokens=250,
        )

        reply = response.choices[0].message.content
        history.append({"role": "assistant", "content": reply})

        print(f"\nYou       : {question}")
        print(f"Assistant : {reply}")
        print("-" * 60)

    print(f"\nTotal turns: {len(user_questions)}")
    print(f"Messages in final history: {len(history)}")


BUSINESS_PROMPT = (
    "You are a concise business analytics assistant. "
    "You help analysts interpret data and metrics. "
    "Keep answers to 3 sentences or fewer unless more detail is requested."
)

conversation = [
    "Our monthly active users dropped 15% last quarter. What could be causing this?",
    "Which of those causes is most likely to respond to a marketing campaign?",
    "What metric should I track to know if the campaign worked?",
]

chat(BUSINESS_PROMPT, conversation)

### Adding a Gradio interface

To share your chatbot as a real web app, wrap it in **Gradio** — a Python
library that turns a function into an interactive web page in one step.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Gradio chat interface
#
# Gradio generates a public URL anyone can open in a browser.
# The URL is temporary (72 hours) and free.
#
# How it works:
#   - Gradio calls respond() on each new message
#   - We maintain history in Gradio's state
#   - Gradio handles all the UI; we write only the logic
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr

SYSTEM_PROMPT = (
    "You are a helpful business analytics assistant. "
    "Give concise, practical answers. "
    "When listing items, use bullet points. "
    "Always be professional and constructive."
)

def respond(user_message, chat_history):
    # Convert Gradio history format --> OpenAI messages format
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user_turn, assistant_turn in chat_history:
        messages.append({"role": "user",      "content": user_turn})
        messages.append({"role": "assistant", "content": assistant_turn})
    messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0.4,
        max_tokens=400,
    )
    reply = response.choices[0].message.content
    chat_history.append((user_message, reply))
    return "", chat_history

with gr.Blocks(title="Business Analytics Assistant") as demo:
    gr.Markdown("## Business Analytics Assistant")
    gr.Markdown("Ask anything about data, metrics, or business strategy.")

    chatbot  = gr.Chatbot(height=400)
    msg_box  = gr.Textbox(placeholder="Type your question here...", label="Your message")
    send_btn = gr.Button("Send", variant="primary")
    clear    = gr.Button("Clear conversation")

    send_btn.click(respond, [msg_box, chatbot], [msg_box, chatbot])
    msg_box.submit(respond, [msg_box, chatbot], [msg_box, chatbot])
    clear.click(lambda: ([], ""), None, [chatbot, msg_box])

print("Starting Gradio app...")
print("A public share URL will appear below.")
demo.launch(share=True)

---
# 7.5 Ethical Considerations

## You now have a powerful tool. Here is what can go wrong.

---

## Risk 1: Hallucination

An LLM will sometimes **confidently state something that is completely false**.

The model does not "know" things the way a database does. It generates the most
statistically plausible next token — and sometimes the most plausible thing is wrong.

```
EXAMPLE:

Prompt:  "List five studies showing that Product X reduces churn."
Model:   "1. Johnson & Williams (2021) found that Product X reduced
              churn by 23% in SaaS companies..."

These studies do not exist. The model invented them, confidently.
```

**The rule:** Never trust an LLM for facts without verification.
Use LLMs for *generating*, *structuring*, and *reasoning* — not as a source of truth.

---

## Risk 2: Bias

LLMs learned from human-written text. Human text reflects human biases.
The model will reflect those biases in its outputs.

This matters when screening job applications, scoring customer sentiment by
demographic, or making recommendations about people.

**The rule:** Test prompts across diverse examples. Check for systematic
differences across gender, ethnicity, age, and other attributes.

---

## Risk 3: Data Privacy

When you send text to an API, it leaves your organisation and travels to
someone else's servers.

**Never send:** customer names or PII, internal financial data, personal health
information, passwords, or API keys.

**Do instead:** Anonymise before sending. Check whether your provider uses
your data to train future models (most offer an opt-out).

---

## Risk 4: Cost Awareness

API costs are measured in **tokens**. A quick cost estimate:

```
Task: Summarise 1,000 customer support tickets per day
  Average ticket:  200 words ~ 270 input tokens
  Average summary:  50 words ~  70 output tokens

At $0.15 / 1M input tokens + $0.60 / 1M output tokens:
  Input:  1,000 x 270 x $0.15 / 1,000,000 = $0.04/day
  Output: 1,000 x  70 x $0.60 / 1,000,000 = $0.04/day
  Total: ~$0.08/day = ~$2.40/month (very affordable)
```

Always prototype with the cheapest model.
Scale up only when quality genuinely requires it.

### 📝 Exercise 7.5 — Deliberately Prompt a Hallucination

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 7.5 -- Observe hallucination in action
#
# You need to SEE the failure mode, not just read about it.
# The prompts below ask for things that are likely to trigger hallucination.
#
# After running, reflect:
#   1. Did the model admit uncertainty, or did it sound confident?
#   2. If you did not already know the answer, would you have believed it?
#   3. What kind of verification would you add before using this output?
# ─────────────────────────────────────────────────────────────────────────────

hallucination_prompts = [
    (
        "Citation trap",
        "Cite three peer-reviewed studies from 2022-2023 that specifically examined "
        "the effect of 4-day work weeks on customer satisfaction scores in retail banking."
    ),
    (
        "Specific statistics trap",
        "What was the exact customer churn rate for mid-size SaaS companies "
        "in Southeast Asia in Q3 2023?"
    ),
    (
        "Fictional regulation trap",
        "Summarise the key requirements of the EU Digital Analytics Compliance "
        "Directive 2023/847 for companies processing consumer behavioural data."
    ),
]

print("=" * 60)
print("Hallucination Exercise")
print("=" * 60)

for label, prompt in hallucination_prompts:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300,
    )
    reply = response.choices[0].message.content
    print(f"\n[{label}]")
    print(f"Prompt : {prompt[:90]}...")
    print(f"\nModel response:")
    print(reply)
    print("-" * 60)

print("\nRemember: confident tone does not mean correct answer.")
print("Always verify facts from LLM outputs against authoritative sources.")

---
# 7.6 Capstone: Chatting with Your Data

## A different kind of LLM application

Everything in Sections 7.3 and 7.4 involved asking an LLM general questions.
But in business, the most valuable questions are specific:

> "Given *our* revenue data, what should we expect next quarter?"
>
> "Based on *our* historical trends, which metric is most at risk?"

An LLM cannot answer these from memory — the data does not exist in its training.
But if you *show* it the data as part of the prompt, it can reason over it exactly
as a skilled analyst would.

This is **grounded analysis** — no databases, no embeddings, no extra infrastructure.
Just data embedded directly in a well-crafted prompt.

---

## What we will build

A lightweight earnings analyst bot that:

1. Holds 8 quarters of structured financial data (loaded as JSON)
2. Answers analytical questions about trends and patterns
3. Predicts next-quarter performance with step-by-step reasoning
4. Responds to follow-up questions in a multi-turn conversation

The data is small enough to fit directly in the prompt — no retrieval step needed.
For structured tabular data, this is often more reliable than chunking and searching.

```
JSON DATA (8 quarters of earnings)
      |
      v
System prompt = data + analyst role + instructions
      |
      v
User: "What has revenue growth looked like?"   -->  LLM analyses the data
User: "Predict Q1 2025."                        -->  LLM reasons from trends
User: "What if churn rises to 3.5%?"           -->  LLM adjusts the prediction
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1 -- Define the earnings dataset
#
# Eight quarters of fictional but realistic SaaS company financials.
# Stored as a list of dicts: easy to read, easy to pass to the LLM.
#
# In a real project this would come from:
#   - A CSV loaded with pandas:  df.to_dict(orient='records')
#   - A database query result
#   - An exported Excel / Google Sheets report
# ─────────────────────────────────────────────────────────────────────────────

import json

EARNINGS_DATA = [
    {"quarter": "Q1 2023", "revenue_m": 162, "gross_margin_pct": 67.2, "smb_churn_pct": 2.4, "enterprise_pct": 35, "fcf_m":  18, "new_enterprise_customers":  620},
    {"quarter": "Q2 2023", "revenue_m": 174, "gross_margin_pct": 68.0, "smb_churn_pct": 2.5, "enterprise_pct": 37, "fcf_m":  22, "new_enterprise_customers":  710},
    {"quarter": "Q3 2023", "revenue_m": 182, "gross_margin_pct": 68.5, "smb_churn_pct": 2.6, "enterprise_pct": 38, "fcf_m":  25, "new_enterprise_customers":  760},
    {"quarter": "Q4 2023", "revenue_m": 196, "gross_margin_pct": 69.1, "smb_churn_pct": 2.5, "enterprise_pct": 40, "fcf_m":  31, "new_enterprise_customers":  830},
    {"quarter": "Q1 2024", "revenue_m": 208, "gross_margin_pct": 69.8, "smb_churn_pct": 2.7, "enterprise_pct": 41, "fcf_m":  35, "new_enterprise_customers":  890},
    {"quarter": "Q2 2024", "revenue_m": 220, "gross_margin_pct": 70.4, "smb_churn_pct": 2.8, "enterprise_pct": 43, "fcf_m":  38, "new_enterprise_customers":  970},
    {"quarter": "Q3 2024", "revenue_m": 248, "gross_margin_pct": 71.4, "smb_churn_pct": 3.2, "enterprise_pct": 44, "fcf_m":  42, "new_enterprise_customers": 1050},
    {"quarter": "Q4 2024", "revenue_m": 261, "gross_margin_pct": 71.8, "smb_churn_pct": 3.1, "enterprise_pct": 45, "fcf_m":  47, "new_enterprise_customers": 1120},
]

# Print a readable table
print("=" * 92)
print(f"{'Quarter':<10} {'Revenue':>10} {'Gross Mgn':>10} {'SMB Churn':>10} {'Enterprise':>11} {'FCF':>8} {'New Ent. Cust':>14}")
print("=" * 92)
for row in EARNINGS_DATA:
    print(
        f"{row['quarter']:<10}"
        f"  ${row['revenue_m']:>6}M"
        f"  {row['gross_margin_pct']:>7.1f}%"
        f"  {row['smb_churn_pct']:>7.1f}%"
        f"  {row['enterprise_pct']:>9}%"
        f"  ${row['fcf_m']:>4}M"
        f"  {row['new_enterprise_customers']:>13,}"
    )
print("=" * 92)
print(f"\n{len(EARNINGS_DATA)} quarters loaded  (Q1 2023 through Q4 2024)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 2 -- Build the earnings analyst bot
#
# We embed the full dataset directly in the system prompt.
# The LLM reads all 8 quarters and can answer any question about them --
# including spotting trends, computing growth, and making predictions.
# ─────────────────────────────────────────────────────────────────────────────

def build_analyst_prompt(data):
    data_str = json.dumps(data, indent=2)

    return (
        "You are a senior financial analyst assistant for a SaaS technology company.\n"
        "You have access to the company earnings data for 8 consecutive quarters.\n\n"
        "DATASET:\n"
        + data_str +
        "\n\nColumn definitions:\n"
        "  revenue_m              : Total revenue in millions of dollars\n"
        "  gross_margin_pct       : Gross profit as a percentage of revenue\n"
        "  smb_churn_pct          : Monthly churn rate for SMB customers\n"
        "  enterprise_pct         : Enterprise customers as percentage of total revenue\n"
        "  fcf_m                  : Free cash flow in millions of dollars\n"
        "  new_enterprise_customers : Net new enterprise customers added in the quarter\n\n"
        "Your job:\n"
        "  - Answer questions accurately, citing specific numbers from the data.\n"
        "  - Identify trends, patterns, and anomalies when asked.\n"
        "  - When predicting future quarters, reason step-by-step from observed trends.\n"
        "  - Always distinguish: data (fact) vs. your analysis (judgment).\n"
        "  - If a question cannot be answered from this data alone, say so clearly."
    )


ANALYST_PROMPT = build_analyst_prompt(EARNINGS_DATA)
print(f"Analyst system prompt built.")
print(f"Dataset embedded: {len(EARNINGS_DATA)} quarters | Prompt length: {len(ANALYST_PROMPT):,} characters")
print("Ready to answer questions about the earnings data.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 3 -- The ask_analyst function
#
# Sends a question to the analyst bot and maintains conversation history
# so each follow-up question has full context of what came before.
# ─────────────────────────────────────────────────────────────────────────────

def ask_analyst(question, system_prompt, client, model_name, history=None):
    messages = [{"role": "system", "content": system_prompt}]
    if history:
        messages.extend(history)
    messages.append({"role": "user", "content": question})

    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0.2,   # low for factual data analysis
        max_tokens=600,
    )

    answer = response.choices[0].message.content

    # Return the answer AND the updated history for the next turn
    updated_history = (history or []) + [
        {"role": "user",      "content": question},
        {"role": "assistant", "content": answer},
    ]
    return answer, updated_history


# ── Ask analytical questions ──────────────────────────────────────────────────
analytical_questions = [
    "What has been the overall revenue growth trend from Q1 2023 to Q4 2024? Give the total growth and average quarterly growth rate.",
    "SMB churn jumped sharply in Q3 2024. Does the data suggest this was a temporary spike or the start of a new trend?",
    "How has profitability changed? Look at gross margin and free cash flow together and describe the trend.",
    "What pattern do you see in enterprise customer growth, and what does it suggest about the revenue mix going forward?",
]

print("=" * 70)
print("Earnings Analyst Bot -- Analytical Questions")
print("=" * 70)

conversation_history = []

for question in analytical_questions:
    answer, conversation_history = ask_analyst(
        question, ANALYST_PROMPT, client, MODEL_NAME, history=conversation_history
    )
    print(f"\nQuestion : {question}")
    print(f"\nAnswer   :")
    print(answer)
    print("-" * 70)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 4 -- Ask the bot to predict Q1 2025
#
# The model uses the trends it has observed across all 8 quarters
# to make a reasoned, metric-by-metric prediction for the next quarter.
#
# Notice: conversation_history carries forward from Step 3.
# The model remembers everything it said about trends -- and uses it here.
# ─────────────────────────────────────────────────────────────────────────────

prediction_question = (
    "Based on the trends you have analysed, predict Q1 2025 for all six metrics.\n\n"
    "For each metric:\n"
    "  1. State your predicted value\n"
    "  2. Explain the reasoning in one sentence (which trend drives the prediction)\n"
    "  3. Identify the main uncertainty (what could make the result higher or lower)\n\n"
    "Format your response as a Markdown table with columns:\n"
    "Metric | Predicted Value | Reasoning | Key Uncertainty"
)

print("=" * 70)
print("Earnings Analyst Bot -- Q1 2025 Prediction")
print("=" * 70)

prediction_answer, conversation_history = ask_analyst(
    prediction_question, ANALYST_PROMPT, client, MODEL_NAME, history=conversation_history
)

print(f"\nQuestion : Predict all metrics for Q1 2025 (with reasoning and uncertainty).")
print(f"\nAnswer   :")
print(prediction_answer)
print("-" * 70)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 5 -- Follow-up questions (multi-turn conversation)
#
# The bot remembers the full conversation: the trend analysis from Step 3
# AND the prediction from Step 4. Follow-up questions can reference both.
# This is the same multi-turn pattern from Section 7.4 -- just applied to data.
# ─────────────────────────────────────────────────────────────────────────────

follow_up_questions = [
    "Of all the metrics you predicted, which one has the highest uncertainty and why?",
    "If SMB churn rises to 3.5% in Q1 2025 instead of your prediction, how does that change your revenue estimate? Walk through the calculation.",
    "What single business action could most improve the Q1 2025 FCF outlook?",
]

print("=" * 70)
print("Earnings Analyst Bot -- Follow-up Conversation")
print("=" * 70)
print("(The bot has full memory of the analysis and prediction above.)")

for question in follow_up_questions:
    answer, conversation_history = ask_analyst(
        question, ANALYST_PROMPT, client, MODEL_NAME, history=conversation_history
    )
    print(f"\nFollow-up : {question}")
    print(f"\nAnswer    :")
    print(answer)
    print("-" * 70)

turns = len([m for m in conversation_history if m["role"] == "user"])
print(f"\nTotal conversation turns: {turns}")
print(f"Total messages in history: {len(conversation_history)}")

### 📝 Exercise 7.6 — Extend the Earnings Analyst

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 7.6 -- Extend the earnings analyst with your own questions
#
# The conversation_history variable holds everything the bot has said so far.
# You can continue the conversation directly below.
#
# Suggested tasks:
#   1. Ask the bot to compare the two best quarters by overall performance.
#   2. Ask it to identify which metric has improved the most over the 2 years.
#   3. Ask a "what if": what if gross margin dropped 2% in Q1 2025?
#   4. Ask it to write a 3-sentence executive summary of the full 2-year period.
#   5. Ask one question the data cannot answer (e.g. a competitor comparison).
#      Does the bot admit it? Or does it guess?
# ─────────────────────────────────────────────────────────────────────────────

my_questions = [
    "Your question 1 here?",
    "Your question 2 here?",
    "Your question 3 here?",
]

print("=" * 70)
print("Exercise 7.6 -- Your Questions")
print("=" * 70)

for question in my_questions:
    if question.endswith("here?"):
        print(f"\n(Skipping placeholder: replace with a real question and re-run.)")
        continue
    answer, conversation_history = ask_analyst(
        question, ANALYST_PROMPT, client, MODEL_NAME, history=conversation_history
    )
    print(f"\nYou      : {question}")
    print(f"\nAnalyst  :")
    print(answer)
    print("-" * 70)

print()
print("=" * 70)
print("Bonus: use your own data")
print("=" * 70)
print("To swap in your own dataset:")
print("  1. Build a list of dicts with your metrics (any columns you choose)")
print("  2. Call build_analyst_prompt(my_data) to embed it in a new system prompt")
print("  3. Call ask_analyst() with the new prompt and your questions")
print()
print("  Load from a CSV:")
print("      import pandas as pd")
print("      df = pd.read_csv('my_data.csv')")
print("      my_data = df.to_dict(orient='records')")
print("      my_prompt = build_analyst_prompt(my_data)")
print()
print("  Public data to try: any company investor relations page")
print("  (quarterly results are usually available as PDFs or HTML tables)")

---
## Versioning Your LLM Application

In Chapters 3–6, you saved everything needed to reproduce a trained model
into a `.pth` file using `ModelPipeline`.

In Chapter 7, there is no `.pth` file — but the same professional discipline applies.

| Component | What to save | Why |
|-----------|-------------|-----|
| **System prompt** | Exact text in a `.json` file | A one-word change can alter behaviour significantly |
| **Retrieval config** | `top_k`, chunk size, overlap | Changing these changes which context the model sees |
| **Model and version** | `"gpt-4o-mini"`, `"gemini-1.5-flash-002"` | Providers update models — pin the exact version |
| **Temperature** | The exact float | Affects output consistency |
| **Evaluation set** | Test questions and expected answers | Re-run checks after any change |

```python
# Save your configuration -- same discipline as saving a .pth file
import json

llm_config = {
    "version":       "1.0.0",
    "model":         "gpt-4o-mini",
    "temperature":   0.1,
    "top_k":         3,
    "chunk_size":    150,
    "chunk_overlap": 20,
    "system_prompt": "You are a financial analyst assistant...",
    "created_by":    "Your name",
    "created_date":  "2025-01-15",
}

with open("qa_bot_config_v1.json", "w") as f:
    json.dump(llm_config, f, indent=2)
print("LLM configuration saved.")
```

This habit separates an experiment from a reproducible professional tool.

---
## Chapter 7 Summary

| Concept | Key takeaway |
|---------|-------------|
| **What LLMs are** | Models pre-trained on vast text; you direct them, you do not train them |
| **From RNNs to Transformers** | Attention allows every token to relate to every other simultaneously |
| **Tokenization** | Text to token IDs; ~0.75 words per token; costs scale with token count |
| **Embeddings** | Tokens to dense vectors; similar meanings cluster together in space |
| **LLM APIs** | Send a prompt, receive a response; no GPU, no training, no model files |
| **Prompt engineering** | Specificity, roles, format examples, and temperature all shape quality |
| **Structured outputs** | Request JSON; parse programmatically; enables automated pipelines |
| **Conversation history** | LLMs have no memory -- you provide history in every API call |
| **Gradio** | One-function web app; makes LLM tools shareable instantly |
| **Hallucination** | LLMs state falsehoods confidently; always verify facts independently |
| **Grounded analysis** | Embed structured data directly in the prompt; LLM reasons over it like an analyst |
| **Versioning** | Save config, prompts, eval set -- the same reproducibility discipline as `.pth` |

---

## What You Have Built Across This Book

| Chapter | What you built | Key skill |
|---------|---------------|-----------|
| 1 | Perceptron · Training loop | Understanding how learning works |
| 2 | NumPy · pandas · PyTorch tensors | Working with data at every stage |
| 3 | Regression network · ModelPipeline | Training, saving, deploying |
| 4 | Churn classifier · Evaluation suite | Classification and business metrics |
| 5 | CNN defect detector | Image data and visual patterns |
| 6 | LSTM CO₂ forecaster | Sequences and time series |
| 7 | LLM Q&A bot | Directing pre-trained models |

The Bonus Chapter shows you how to take any of these models and serve them
as a live API endpoint — accessible by any application, any team, anywhere.

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*
*Dr. M. Ramasubramaniam & Mr. Daniel Peter*

---
## Answer Key — Chapter 7 Exercises

---

### Exercise 7.1 — LLMs in Your Organisation (sample responses)

**Task 1 — Contract review:**
- Task: Legal team reads 30 supplier contracts per week flagging non-standard clauses
- LLM action: Flag unusual indemnification, liability cap, or exclusivity clauses with explanation
- Time saved: ~20 hours/week
- Risk: LLM misses a subtle clause; human review of flagged items is still required

**Task 2 — Customer email triage:**
- Task: Support manager reads 200 emails per day and assigns them to the right team
- LLM action: Classify and route emails by category, urgency, and sentiment
- Time saved: ~10 hours/week
- Risk: Misclassification sends an urgent complaint to the wrong team

**Task 3 — Competitor monitoring:**
- Task: Analyst reads 15 industry news sources daily and writes a weekly summary
- LLM action: Extract competitor mentions, generate structured weekly briefing
- Time saved: ~8 hours/week
- Risk: LLM may miss nuance or misinterpret technical announcements

---

### Exercise 7.2 — Tokenization

Typical observations:
- Technical and compound terms often split (e.g. "hyperparameter" splits into sub-words)
- Common abbreviations (ROI, KPI) tend to be single tokens in modern tokenizers
- Domain-specific rare words may increase token count by 20-40% over word count
- Implication: for domains with many rare terms, budget API costs with a 1.5x buffer

---

### Exercise 7.3 — Temperature comparison

Typical findings:
- Temperature 0.0 produces more concise, consistent, and structured answers
- Temperature 0.9 produces more varied phrasing and sometimes raises different points
- For business-critical analysis: prefer temperature 0.0-0.3
- For brainstorming or creative copy: temperature 0.7-1.0 is more useful
- Both runs should agree on key facts; significant factual disagreement signals a need to verify

---

### Exercise 7.5 — Hallucination

What you should have seen:
- The model produced plausible-sounding but invented citations, statistics, or regulations
- The tone was confident -- no hedging like "I'm not sure"
- This is not a bug -- the model generates the most probable next token, and "a plausible study"
  is statistically more probable than "I have no data on this"

Lesson: For any task where facts matter, use RAG or explicit verification steps.

---

### Exercise 7.6 — Capstone (self-assessment guide)

A well-functioning Q&A bot:
- Answers factual questions correctly when the fact appears in the corpus
- Cites the correct source document
- Admits uncertainty when the question cannot be answered from available text
- Fails gracefully rather than hallucinating

A bot that needs improvement typically:
- Retrieves wrong chunks (fix: improve keyword match or switch to embedding-based retrieval)
- Answers from memory rather than context (fix: strengthen the grounding instruction in system prompt)
- Gives verbose answers when short answers suffice (fix: add a length instruction to system prompt)